# Institutional Risk Engine Demo

This notebook demonstrates RiskOptima's institutional analytics layer using deterministic synthetic data. It covers option book valuation, Greek aggregation, constrained portfolio optimization, risk attribution, and stress scenarios.

In [2]:
from datetime import date

import numpy as np
import pandas as pd

from riskoptima.options import OptionBook, OptionContract, option_scenario_grid
from riskoptima.optim import Constraints, optimize_max_sharpe
from riskoptima.risk import (
    StressScenario,
    component_cvar,
    component_volatility_contribution,
    factor_risk_contribution,
    run_scenario_set,
    run_stress_scenario,
)

## 1. Value an option book

The option analytics layer values individual contracts and aggregates book-level exposures. Units are explicit: market value is `price * quantity * multiplier`, and notional delta is `delta * spot * quantity * multiplier`.

In [4]:
valuation_date = date(2026, 7, 1)
expiry = date(2027, 7, 1)

contracts = [
    OptionContract("SPY", strike=500, expiry=expiry, option_type="call", quantity=3, implied_volatility=0.22),
    OptionContract("SPY", strike=470, expiry=expiry, option_type="put", quantity=-2, implied_volatility=0.26),
    OptionContract("QQQ", strike=430, expiry=expiry, option_type="call", quantity=1, implied_volatility=0.25),
]

book = OptionBook(contracts)
detail = book.value({"SPY": 505.0, "QQQ": 425.0}, valuation_date=valuation_date, risk_free_rate=0.04)
summary = book.aggregate(detail)

detail[["underlying", "option_type", "strike", "price", "market_value", "notional_delta", "vega_exposure"]]

,underlying,option_type,strike,price,market_value,notional_delta,vega_exposure
0,SPY,call,500,56.561154,16968.346121,95741.826279,57102.435309
1,SPY,put,470,27.231577,-5446.315456,29058.302301,-34443.715273
2,QQQ,call,430,47.883317,4788.331713,25251.084057,16480.735433


In [5]:
pd.Series(summary["total"]).round(2)

market_value       16310.36
notional_delta    150051.21
delta_exposure       306.54
gamma_exposure         0.86
theta_exposure     -9213.31
vega_exposure      39139.46
rho_exposure      133740.85
dtype: float64

## 2. Run an option scenario grid

Scenario grids make option convexity visible by repricing across spot and volatility shocks.

In [7]:
scenario_grid = option_scenario_grid(
    contracts[0],
    spot=505.0,
    valuation_date=valuation_date,
    risk_free_rate=0.04,
    volatility=0.22,
    spot_shocks=(-0.10, -0.05, 0.0, 0.05, 0.10),
    volatility_shocks=(-0.05, 0.0, 0.05),
)
scenario_grid.pivot(index="spot_shock", columns="volatility_shock", values="pnl").round(2)

volatility_shock,-0.05,0.00,0.05
spot_shock,,,
-0.10,-10854.47,-8183.49,-5479.06
-0.05,-7308.92,-4449.87,-1598.64
0.00,-2837.19,0.00,2864.55
0.05,2458.20,5097.17,7858.68
0.10,8431.06,10753.40,13322.75


## 3. Optimize a portfolio with institutional constraints

The optimizer still returns `pd.Series` by default for backwards compatibility. With `return_result=True`, it returns diagnostics, objective values, and active constraints.

In [9]:
assets = ["Equity", "Quality", "Duration", "Gold"]
expected_returns = pd.Series([0.09, 0.07, 0.035, 0.04], index=assets)
cov = pd.DataFrame(
    [
        [0.040, 0.018, -0.004, 0.006],
        [0.018, 0.030, -0.002, 0.004],
        [-0.004, -0.002, 0.010, 0.001],
        [0.006, 0.004, 0.001, 0.025],
    ],
    index=assets,
    columns=assets,
)
previous_weights = pd.Series(np.repeat(0.25, 4), index=assets)
factor_exposures = pd.DataFrame({"growth": [1.0, 0.6, -0.2, 0.1]}, index=assets)
metadata = pd.DataFrame(
    {"sector": ["Cyclical", "Defensive", "Rates", "Commodity"], "asset_class": ["Equity", "Equity", "Bond", "Commodity"]},
    index=assets,
)
constraints = Constraints(
    weight_bounds=(0.0, 0.55),
    leverage_limit=1.0,
    turnover_limit=0.35,
    factor_bounds={"growth": (0.0, 0.65)},
    asset_class_bounds={"Bond": (0.10, 0.45)},
)

opt_result = optimize_max_sharpe(
    expected_returns,
    cov,
    constraints=constraints,
    previous_weights=previous_weights,
    factor_exposures=factor_exposures,
    metadata=metadata,
    return_result=True,
)

opt_result.weights.round(4)

Equity      0.2461
Quality     0.1918
Duration    0.4250
Gold        0.1371
dtype: float64

In [10]:
pd.Series({
    "expected_return": opt_result.expected_return,
    "volatility": opt_result.volatility,
    "sharpe": opt_result.sharpe,
    "success": opt_result.success,
}).to_frame("value")

,value
expected_return,0.055935
volatility,0.084089
sharpe,0.665191
success,True


## 4. Attribute portfolio risk

Component contributions decompose portfolio risk into asset or factor sources. This is useful for explaining why a portfolio is risky, not only how risky it is.

In [12]:
vol_contrib = component_volatility_contribution(opt_result.weights, cov)
factor_cov = pd.DataFrame([[0.035]], index=["growth"], columns=["growth"])
factor_contrib = factor_risk_contribution(opt_result.weights, factor_exposures[["growth"]], factor_cov)

pd.concat([vol_contrib.rename("vol_contribution"), opt_result.weights.rename("weight")], axis=1).round(5)

,vol_contribution,weight
Equity,0.03635,0.24613
Quality,0.02254,0.19179
Duration,0.01526,0.42500
Gold,0.00994,0.13708


In [13]:
rng = np.random.default_rng(42)
synthetic_returns = pd.DataFrame(rng.multivariate_normal(expected_returns / 252, cov / 252, size=500), columns=assets)
component_cvar(synthetic_returns, opt_result.weights, confidence=0.95).round(6)

Equity      0.005099
Quality     0.002718
Duration    0.001301
Gold        0.001111
Name: component_cvar, dtype: float64

## 5. Run stress scenarios

The scenario engine applies direct price shocks or factor shocks to holdings. The outputs are deterministic and suitable for tests, dashboards, or reports.

In [15]:
holdings = pd.Series({"Equity": 100.0, "Quality": 80.0, "Duration": 60.0, "Gold": 40.0})
prices = pd.Series({"Equity": 100.0, "Quality": 95.0, "Duration": 90.0, "Gold": 120.0})
scenario = StressScenario("risk_off", price_shocks={"Equity": -0.18, "Quality": -0.10, "Gold": 0.08})
result = run_stress_scenario(holdings, prices, scenario)

print(result)
result.asset_pnl.round(2)

StressResult(scenario=StressScenario(name='risk_off', price_shocks={'Equity': -0.18, 'Quality': -0.1, 'Gold': 0.08}, factor_shocks={}, description=''), base_value=27800.0, stressed_value=25624.0, pnl=-2176.0, pnl_pct=-0.07827338129496403, asset_pnl=Equity     -1800.0
Quality     -760.0
Duration       0.0
Gold         384.0
Name: asset_pnl, dtype: float64)


Equity     -1800.0
Quality     -760.0
Duration       0.0
Gold         384.0
Name: asset_pnl, dtype: float64

In [16]:
scenarios = [
    StressScenario("equity_crash", price_shocks={"Equity": -0.20, "Quality": -0.12}),
    StressScenario("rates_up", price_shocks={"Duration": -0.08}),
    StressScenario("gold_rally", price_shocks={"Gold": 0.10}),
]
run_scenario_set(holdings, prices, scenarios).round(4)

,base_value,stressed_value,pnl,pnl_pct
scenario,,,,
equity_crash,27800.0,24888.0,-2912.0,-0.1047
rates_up,27800.0,27368.0,-432.0,-0.0155
gold_rally,27800.0,28280.0,480.0,0.0173
